# ALFRED DFII10 初次发布值来源探测

冻结一次只读公开表单下载，保留精确响应和 NAS 回读；不读取 SOXX 结果、拟合或运行账户。

In [1]:
import csv,hashlib,io,json,tempfile,zipfile
from datetime import datetime,timezone,date
from pathlib import Path
from urllib.parse import urljoin
import requests
from src.data.nas_market_data import publish_nas_snapshot,materialize_nas_snapshot

root=Path.cwd();card=root/'research/stage1_treasury_real_yield_alfred_initial_20261023'
plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists()
raw=root/'data/snapshots/stage1_treasury_real_yield_alfred_initial_20261023';raw.mkdir(parents=True,exist_ok=True);assert not any(raw.iterdir())
state={'status':'started','plan_sha256':sha(card/'plan.json'),'POST_attempts':0,'redirect_GETs':0,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[];failure=None;download=None;parsed=None
session=requests.Session();session.trust_env=False;session.headers.update({'User-Agent':plan['request_policy']['user_agent']})
policy=plan['request_policy']
def capture(method,url,step,form=None):
    started=datetime.now(timezone.utc).isoformat();receipt={'method':method,'url':url,'requested_at_utc':started,'step':step}
    body_path=raw/(step+'.body');header_path=raw/(step+'.headers.json')
    try:
        response=session.request(method,url,data=form,timeout=(policy['connect_timeout_seconds'],policy['total_timeout_seconds']),allow_redirects=False,stream=True)
        entity=response.raw.read(policy['max_response_bytes']+1,decode_content=False)
        body_path.write_bytes(entity)
        headers={'status_code':response.status_code,'url':response.url,'headers':dict(response.headers)}
        header_path.write_text(json.dumps(headers,indent=2)+'\n')
        receipt.update(received_at_utc=datetime.now(timezone.utc).isoformat(),http_status=response.status_code,final_url=response.url,body_sha256=sha(body_path),body_bytes=len(entity),headers_sha256=sha(header_path),headers_bytes=header_path.stat().st_size,content_type=response.headers.get('Content-Type'),location=response.headers.get('Location'))
        response.close()
    except Exception as exc:
        receipt.update(received_at_utc=datetime.now(timezone.utc).isoformat(),transport_exception=type(exc).__name__+':'+str(exc)[:220],body_sha256=sha(body_path) if body_path.exists() else None,body_bytes=body_path.stat().st_size if body_path.exists() else 0)
    receipts.append(receipt)
    (raw/'capture_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
    return receipt,body_path
state['POST_attempts']=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
first,first_body=capture('POST',plan['exact_post']['url'],'post',plan['exact_post']['fields'])
final,download=first,first_body
if first.get('http_status') in (301,302,303,307,308) and first.get('location') and not first.get('transport_exception'):
    state['redirect_GETs']=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
    final,download=capture('GET',urljoin(first['final_url'],first['location']),'redirect')
if final.get('transport_exception') or final.get('http_status')!=200 or final.get('body_bytes',0)>policy['max_response_bytes']:
    failure='transport_status_or_size'
elif not zipfile.is_zipfile(download):
    failure='not_ZIP:'+str(final.get('content_type'))
else:
    try:
        with zipfile.ZipFile(download) as archive:
            members=[m for m in archive.infolist() if not m.is_dir()]
            assert 1<=len(members)<=10 and all(m.file_size<=20000000 and '..' not in Path(m.filename).parts for m in members)
            csv_members=[m for m in members if m.filename.lower().endswith('.csv')]
            assert len(csv_members)==1,[m.filename for m in members]
            csv_bytes=archive.read(csv_members[0]);(raw/'initial_release.csv').write_bytes(csv_bytes)
        rows=list(csv.reader(io.StringIO(csv_bytes.decode('utf-8-sig'))))
        header=rows[0];assert len(rows)>1000 and len(header)>=2
        parsed={'zip_members':[m.filename for m in members],'csv_member':csv_members[0].filename,'csv_header':header,'csv_rows':len(rows)-1,'first_data_row':rows[1][:5],'last_data_row':rows[-1][:5],'csv_sha256':sha(raw/'initial_release.csv')}
    except Exception as exc:
        failure='ZIP_or_CSV_schema:'+type(exc).__name__+':'+str(exc)[:160]
manifest={'schema_version':1,'experiment':plan['experiment'],'plan_sha256':state['plan_sha256'],'capture_utc':datetime.now(timezone.utc).isoformat(),'request_receipts':receipts,'parsed':parsed,'failure':failure,'initial_release_semantics_requested_by_form':True,'historical_first_public_clock_qualified':False}
(raw/'source_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
(card/'source_receipt.json').write_text(json.dumps(manifest,indent=2)+'\n')
files={str(p.relative_to(root)):'raw' for p in raw.iterdir() if p.is_file()}
try:
    binding=publish_nas_snapshot(workspace_root=root,source_manifest=raw/'source_manifest.json',files=files,as_of=date.today().isoformat(),source_metadata={'experiment':plan['experiment'],'source':'ALFRED DFII10 initial release form response'})
    (card/'nas_binding.json').write_text(json.dumps(binding,indent=2)+'\n')
    with tempfile.TemporaryDirectory(prefix='alfred-cold-') as temp:
        readback=materialize_nas_snapshot(binding,workspace_root=Path(temp))
        for obj in binding['objects']:assert sha(Path(temp)/obj['local_path'])==obj['sha256']
    assert binding['verified'] and readback['restored_objects']==len(files)
except Exception as exc:
    failure=(failure+';' if failure else '')+'NAS:'+type(exc).__name__+':'+str(exc)[:180]
    binding=None
state.update(status='completed' if not failure else 'stopped',failure=failure)
(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
result={'decision':'ALFRED_DFII10_initial_release_response_received_clock_unqualified' if not failure else 'needs_ALFRED_download_or_NAS','failure':failure,'POST_attempts':state['POST_attempts'],'redirect_GETs':state['redirect_GETs'],'final_http_status':final.get('http_status'),'final_content_type':final.get('content_type'),'parsed':parsed,'NAS_source_id':binding['source_id'] if binding else None,'NAS_readback_verified':bool(binding),'historical_first_public_clock_qualified':False,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
print(json.dumps(result))


{"decision": "needs_ALFRED_download_or_NAS", "failure": "transport_status_or_size", "POST_attempts": 1, "redirect_GETs": 0, "final_http_status": null, "final_content_type": null, "parsed": null, "NAS_source_id": "0c1eeb955a9022ce1f769e82743d0b26e000aad67f95ea953e5000b407b0f3f3", "NAS_readback_verified": true, "historical_first_public_clock_qualified": false, "SOXX_outcome_reads": 0, "model_fits": 0, "native_accounts": 0}


# ALFRED DFII10 初次发布值：固定表单尝试失败

按冻结计划仅提交一次公开的“Initial Release Only／Zipped CSV”表单，范围 2018-10-01 至 2026-10-07。HTTPS 请求在等待响应 **60 秒后 ReadTimeout**，没有收到 HTTP 状态、响应头或数据体；没有重试或改表单参数。请求时钟、字段、异常和空响应清单已写入 `source_receipt.json`，并在 Yanxuan NAS 做远端哈希及定向回读。

因此本固定传输版本的结论是 `needs_ALFRED_download_or_NAS`，具体依赖是公开表单 POST 在当前传输路径无响应。官方[ALFRED 下载页](https://alfred.stlouisfed.org/series/downloaddata?seid=DFII10)说明可以按历史日期检索当时版本，但网页说明不等于已取得原始值，更不证明具体发布时间。原财政部合并 CSV 仍只有回溯来源资格，不能拟合、运行完整 Stage1 0bp 账户或声称超过 SOXX。

这次 1 次 POST、0 重定向 GET、0 SOXX 结果读取、0 拟合、0 账户。若继续寻找历史初值，必须另冻不同的官方只读传输路径；不得在本固定版本上反复提交或更改日期与输出格式。
